## 1. Imports

In [35]:
import os
import re
import numpy as np
import tensorflow as tf
from tensorflow import keras

import sentencepiece as spm
import kagglehub

print("TF:", tf.__version__)
print(tf.config.list_physical_devices("GPU"))

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

TF: 2.16.2
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Dataset

In [36]:
path = kagglehub.dataset_download("rohitgr/wikitext")
wikitext2_path = os.path.join(path, "wikitext-2")

def load_lines(fp):
    with open(fp, "r", encoding="utf-8") as f:
        return f.readlines()

train_lines = load_lines(os.path.join(wikitext2_path, "wiki.train.tokens"))
val_lines   = load_lines(os.path.join(wikitext2_path, "wiki.valid.tokens"))
test_lines  = load_lines(os.path.join(wikitext2_path, "wiki.test.tokens"))

## 3. Clean data

In [37]:
def clean(x):
    x = re.sub(r"^\s*=+\s*[^=\n]+?\s*=+\s*$", "", x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

train_lines = [clean(x) for x in train_lines if clean(x)]
val_lines   = [clean(x) for x in val_lines if clean(x)]
test_lines  = [clean(x) for x in test_lines if clean(x)]

## 4. SentencePiece tokenizer

In [38]:
with open("train.txt", "w", encoding="utf-8") as f:
    for x in train_lines:
        f.write(x + "\n")

vocab_size = 8000

spm.SentencePieceTrainer.train(
    input="train.txt",
    model_prefix="spm",
    vocab_size=vocab_size,
    model_type="bpe",
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)

sp = spm.SentencePieceProcessor()
sp.load("spm.model")

eos = sp.eos_id()

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: train.txt
  input_format: 
  model_prefix: spm
  model_type: BPE
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differential_priva

## 5. Encode

In [39]:
train_seq = [sp.encode(x) + [eos] for x in train_lines]
val_seq   = [sp.encode(x) + [eos] for x in val_lines]
test_seq  = [sp.encode(x) + [eos] for x in test_lines]

train_tokens = np.concatenate(train_seq).astype(np.int32)
val_tokens   = np.concatenate(val_seq).astype(np.int32)
test_tokens  = np.concatenate(test_seq).astype(np.int32)

vocab_size = sp.get_piece_size()

## 6. Dataset

In [40]:
seq_len = 128
batch_size = 64

def make_dataset(tokens, stride=8, shuffle=False):
    ds = tf.keras.utils.timeseries_dataset_from_array(
        tokens,
        None,
        sequence_length=seq_len + 1,
        sequence_stride=stride,
        batch_size=batch_size,
        shuffle=shuffle
    )

    return ds.map(lambda x: (x[:, :-1], x[:, 1:])).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_tokens, stride=8, shuffle=True)
val_ds   = make_dataset(val_tokens, stride=16)

## 7. Model
- Positional + token embedding

In [41]:
class TokenAndPositionEmbedding(tf.keras.layers.Layer):
    def __init__(self, vocab_size, seq_len, embed_dim):
        super().__init__()
        self.token_emb = keras.layers.Embedding(vocab_size, embed_dim)
        self.pos_emb   = keras.layers.Embedding(seq_len, embed_dim)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        pos = tf.range(seq_len)
        pos = self.pos_emb(pos)

        x = self.token_emb(x)
        return x + pos

- Transformer block (decoder-style)

In [42]:
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        self.att = keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout
        )

        self.ffn = keras.Sequential([
            keras.layers.Dense(ff_dim, activation="relu"),
            keras.layers.Dense(embed_dim),
        ])

        self.norm1 = keras.layers.LayerNormalization()
        self.norm2 = keras.layers.LayerNormalization()
        self.drop1 = keras.layers.Dropout(dropout)
        self.drop2 = keras.layers.Dropout(dropout)

    def call(self, x, training=False):
        seq_len = tf.shape(x)[1]

        causal_mask = tf.linalg.band_part(
            tf.ones((seq_len, seq_len)), -1, 0
        )

        attn = self.att(x, x, attention_mask=causal_mask, training=training)
        x = self.norm1(x + self.drop1(attn, training=training))

        ffn = self.ffn(x)
        x = self.norm2(x + self.drop2(ffn, training=training))

        return x

## 8. Model

In [43]:
embedding_dim = 256
num_heads = 4
layers_n = 2

inputs = keras.Input(shape=(None,), dtype="int32")

x = TokenAndPositionEmbedding(vocab_size, seq_len, embedding_dim)(inputs)

for _ in range(layers_n):
    x = TransformerBlock(
        embedding_dim,
        num_heads,
        embedding_dim * 2
    )(x)

x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(vocab_size)(x)

model = keras.Model(inputs, outputs)
model.summary()

Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_9 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_3  │ (None, None, 256)      │     2,080,768 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_6             │ (None, None, 256)      │       527,104 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_7             │ (None, None, 256)      │       527,104 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_27 (Dropout)            │ (None, None, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, None, 8000)     │     2,056,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,190,976 (19.80 MB)

 Trainable params: 5,190,976 (19.80 MB)

 Non-trainable params: 0 (0.00 B)

## 9. Training

In [44]:
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.AdamW(3e-4, weight_decay=1e-4),
    metrics=[]
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5


W0000 00:00:1778078642.282928  168640 assert_op.cc:38] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
I0000 00:00:1778078645.322756  178042 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_31', 164 bytes spill stores, 164 bytes spill loads

I0000 00:00:1778078645.426288  178045 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_16', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1778078645.682503  178047 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_16', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1778078645.712214  178033 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_31', 156 bytes spill stores, 120 bytes spill loads

I0000 00:00:1778078645.859588  178036 asm_

5320/5321 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 5.0028

W0000 00:00:1778078757.283654  168637 assert_op.cc:38] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
I0000 00:00:1778078761.080817  178811 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_16', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1778078761.380131  178816 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_31', 164 bytes spill stores, 164 bytes spill loads

I0000 00:00:1778078761.583398  178820 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_16', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1778078761.591466  178807 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_16', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1778078761.692433  178819 asm_comp

5321/5321 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 5.0027

W0000 00:00:1778078766.703449  168636 assert_op.cc:38] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
W0000 00:00:1778078769.268433  168638 assert_op.cc:38] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert


5321/5321 ━━━━━━━━━━━━━━━━━━━━ 131s 22ms/step - loss: 4.4142 - val_loss: 3.8283
Epoch 2/5
5321/5321 ━━━━━━━━━━━━━━━━━━━━ 109s 20ms/step - loss: 3.7035 - val_loss: 3.7643
Epoch 3/5
5321/5321 ━━━━━━━━━━━━━━━━━━━━ 106s 20ms/step - loss: 3.4916 - val_loss: 3.7698
Epoch 4/5
5321/5321 ━━━━━━━━━━━━━━━━━━━━ 109s 20ms/step - loss: 3.3749 - val_loss: 3.7818
Epoch 5/5
5321/5321 ━━━━━━━━━━━━━━━━━━━━ 108s 20ms/step - loss: 3.2987 - val_loss: 3.8026


## 10. Sampling

In [48]:
def sample(logits, temp=1.0):
    logits = logits / temp
    logits = logits - np.max(logits)
    p = np.exp(logits)
    return p / np.sum(p)

def generate(text, n=40, temp=0.8):
    ids = sp.encode(text)

    for _ in range(n):
        x = np.array([ids[-seq_len:]], dtype=np.int32)

        logits = model.predict(x, verbose=0)[0, -1]
        probs = sample(logits, temp)

        next_id = np.random.choice(len(probs), p=probs)
        ids.append(int(next_id))

    return sp.decode(ids)

print(generate("the capital of france is"))


the capital of france is dated to the pre @-@ Islamic  ⁇ unk ⁇  . He wrote an early process of Van  ⁇ unk ⁇  , "  ⁇ unk ⁇  , the words of all ruling , " the 
